In [1]:
!pip install faiss-cpu langchain langchain-community langchain-openai pandas python-dotenv

In [1]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from pathlib import Path
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')
llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

import os
os.makedirs('data', exist_ok=True)

import pandas as pd
file_path = ('C:\\Users\\User\\Documents\\sample_data_uk_pet_insurance_quotes (1).csv') 
data = pd.read_csv(file_path)

In [2]:
data.head()

,Id,RiskId,Provider,Species,AnnualPremium,ClaimLimit,AnnualClaimLimit,ExcessLimit,ExcessCoPaymentLimit,UkOnlyCallCentres,...,AdjustmentFee,DuplicateDocumentsFee,CancellationFee,RenewalFee,TimeStamp,VetFees,VetFeesExcess,MonthlyPremiumInstalment,NumberOfMonthlyInstalments,InstalmentDeposit
0,17083559,FIX3946,4PawsPetInsurance,Dog,323.95,No limit,"£4,000",£105,30%,True,...,0,0,0,15,05/03/2023 12:06,NaN,NaN,NaN,NaN,NaN
1,282047527,FIX3946,Animal Friends,Dog,112.08,"£2,000","£7,000",£159,NaN,True,...,0,0,0,0,09/11/2024 00:23,"Covered for up to £2,000 per condition with £7...",You'll pay £159 per condition,9.34,12.0,0.0
2,282047528,FIX3946,Animal Friends,Dog,115.56,"£2,000",Unlimited,£199,NaN,True,...,0,0,0,0,09/11/2024 00:23,"Covered each year up to £2,000 per condition w...",You'll pay £199 per condition per year,9.63,12.0,0.0
3,282047529,FIX3946,Animal Friends,Dog,179.40,"£4,000",Unlimited,£199,NaN,True,...,0,0,0,0,09/11/2024 00:23,"Covered each year up to £4,000 per condition w...",You'll pay £199 per condition per year,14.95,12.0,0.0
4,282047530,FIX3946,Animal Friends,Dog,174.60,"£4,000","£10,000",£159,NaN,True,...,0,0,0,0,09/11/2024 00:23,"Covered for up to £4,000 per condition with £1...",You'll pay £159 per condition,14.55,12.0,0.0


In [3]:
loader = CSVLoader(file_path=file_path)
docs = loader.load_and_split()

In [8]:
print("Documents:", len(docs))
print("Largest document:", max(len(doc.page_content) for doc in docs))

Documents: 7180
Largest document: 971


In [6]:
import faiss
from langchain_openai import OpenAIEmbeddings
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
dimension = len(embeddings.embed_query("test"))

index = faiss.IndexFlatL2(dimension)
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

In [9]:
batch_size = 100

for i in range(0, len(docs), batch_size):
    batch = docs[i:i + batch_size]
    vector_store.add_documents(batch)
    print(f"Added {min(i + batch_size, len(docs))}/{len(docs)} documents")

Added 100/7180 documents
Added 200/7180 documents
Added 300/7180 documents
Added 400/7180 documents
Added 500/7180 documents
Added 600/7180 documents
Added 700/7180 documents
Added 800/7180 documents
Added 900/7180 documents
Added 1000/7180 documents
Added 1100/7180 documents
Added 1200/7180 documents
Added 1300/7180 documents
Added 1400/7180 documents
Added 1500/7180 documents
Added 1600/7180 documents
Added 1700/7180 documents
Added 1800/7180 documents
Added 1900/7180 documents
Added 2000/7180 documents
Added 2100/7180 documents
Added 2200/7180 documents
Added 2300/7180 documents
Added 2400/7180 documents
Added 2500/7180 documents
Added 2600/7180 documents
Added 2700/7180 documents
Added 2800/7180 documents
Added 2900/7180 documents
Added 3000/7180 documents
Added 3100/7180 documents
Added 3200/7180 documents
Added 3300/7180 documents
Added 3400/7180 documents
Added 3500/7180 documents
Added 3600/7180 documents
Added 3700/7180 documents
Added 3800/7180 documents
Added 3900/7180 docum

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

retriever = vector_store.as_retriever()

# Set up system prompt
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    
])     

In [12]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [13]:
answer= rag_chain.invoke({"input": "which company does sheryl Baxter work for?"})
answer['answer']

"Sheryl Baxter works for Sainsbury's."